# 01. DATA INTEGRATION - THU THẬP VÀ TÍCH HỢP DỮ LIỆU
 Data Preparation (Chuẩn bị dữ liệu)

**Mục tiêu:** 
1. Tải tập dữ liệu gốc từ Kaggle.
2. Hợp nhất dữ liệu lịch sử (2009-2023) và dữ liệu mới (2025).
3. Làm giàu dữ liệu bằng cách truy vấn 12 thuộc tính âm học (Audio Features) từ Spotify API.

In [ ]:
import pandas as pd
import kagglehub
import os
import shutil
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import time
import warnings
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
print("--- Thư viện đã sẵn sàng ---")

## 1. Tải dữ liệu từ Kaggle

In [ ]:
# Tải dataset
path = kagglehub.dataset_download("wardabilal/spotify-global-music-dataset-20092025")
print(f"Dataset đã tải về tại: {path}")

# Di chuyển vào thư mục dự án data/raw
raw_folder = "../data/raw/"
os.makedirs(raw_folder, exist_ok=True)

for file in os.listdir(path):
    if file.endswith(".csv"):
        shutil.copy(os.path.join(path, file), raw_folder)
        print(f"Đã sao chép: {file}")

## 2. Hợp nhất các tập dữ liệu gốc

In [ ]:
# Đọc dữ liệu
df_2025 = pd.read_csv('../data/raw/spotify_data clean.csv')
df_history = pd.read_csv('../data/raw/track_data_final.csv')

# Gộp thô
df_merged = pd.concat([df_2025, df_history], ignore_index=True)

# Lọc các track_id duy nhất để tối ưu hóa việc gọi API
unique_tracks = df_merged.drop_duplicates(subset=['track_id'])
print(f"Số lượng bản ghi sau khi gộp: {len(df_merged)}")
print(f"Số lượng ID duy nhất cần truy vấn API: {len(unique_tracks)}")

## 3. Truy vấn Spotify Web API - Lấy Audio Features


In [ ]:
# Bước 1: Tải biến môi trường và kiểm tra credentials
from dotenv import load_dotenv
import requests
import time

# Force reload .env (bỏ qua cache cũ)
load_dotenv(override=True)

client_id = os.getenv('SPOTIPY_CLIENT_ID')
client_secret = os.getenv('SPOTIPY_CLIENT_SECRET')

print("="*70)
print("SPOTIFY WEB API - LẤY AUDIO FEATURES")
print("="*70)

if not client_id or not client_secret:
    print("❌ LỖI: Không tìm thấy SPOTIPY_CLIENT_ID hoặc SPOTIPY_CLIENT_SECRET")
    print("\n📝 HƯỚNG DẪN SETUP:")
    print("1. Vào https://developer.spotify.com/dashboard")
    print("2. Tạo ứng dụng mới")
    print("3. Copy Client ID & Secret")
    print("4. Tạo file .env trong thư mục Spotify-DataMining/ với nội dung:")
    print("   SPOTIPY_CLIENT_ID=your_id")
    print("   SPOTIPY_CLIENT_SECRET=your_secret")
    print("5. Chạy lại cell này")
else:
    print(f"✅ Tìm thấy credentials")
    print(f"   Client ID: {client_id[:10]}...{client_id[-5:]}")
    print(f"   Client Secret: {client_secret[:10]}...{client_secret[-5:]}")

In [ ]:
# Bước 2: Lấy Access Token từ Spotify
def get_spotify_token(client_id, client_secret):
    """Lấy access token từ Spotify OAuth2"""
    auth_url = 'https://accounts.spotify.com/api/token'
    
    auth_data = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
    }
    
    try:
        response = requests.post(auth_url, data=auth_data, timeout=10)
        response.raise_for_status()
        json_result = response.json()
        return json_result['access_token'], json_result['expires_in']
    except requests.exceptions.RequestException as e:
        print(f"❌ Lỗi lấy token: {e}")
        return None, None

# Kiểm tra nếu có credentials
if client_id and client_secret:
    print("\n🔐 Đang lấy Access Token...")
    token, expires_in = get_spotify_token(client_id, client_secret)
    
    if token:
        print(f"✅ Lấy token thành công!")
        print(f"   Token: {token[:30]}...{token[-10:]}")
        print(f"   Hạn sử dụng: {expires_in} giây")
    else:
        print("❌ Không thể lấy token")

In [ ]:

# === PHƯƠNG PHÁP 2: AUTHORIZATION CODE FLOW ===
# Phương pháp này cho phép truy cập dữ liệu cá nhân người dùng (không cần thêm user vào Dashboard)

import webbrowser
from urllib.parse import urlencode, parse_qs
from http.server import HTTPServer, BaseHTTPRequestHandler
import threading
import base64

print("\n" + "="*70)
print("SPOTIFY AUTHORIZATION CODE FLOW - LẤY USER ACCESS TOKEN")
print("="*70)

# ===== CẤU HÌNH =====
REDIRECT_URI = "http://127.0.0.1:8888/callback"
SCOPES = "user-library-read user-read-private user-read-email"

print(f"\n📋 CẤU HÌNH:")
print(f"   Client ID: {client_id[:10] if client_id else 'CHƯA SETUP'}...")
print(f"   Redirect URI: {REDIRECT_URI}")
print(f"   Scopes: {SCOPES}")

# ===== Bước 1: Tạo URL Authorization =====
def get_authorization_url(client_id, redirect_uri, scopes):
    """Tạo URL để người dùng cấp quyền"""
    auth_url = "https://accounts.spotify.com/authorize"
    params = {
        "client_id": client_id,
        "response_type": "code",
        "redirect_uri": redirect_uri,
        "scope": scopes,
        "show_dialog": "true"  # Luôn hiển thị dialog login
    }
    return f"{auth_url}?{urlencode(params)}"

# ===== Bước 2: Server để nhận callback =====
auth_code = None
server_ready = False

class CallbackHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        global auth_code
        
        # Parse URL
        if self.path.startswith("/callback"):
            query = self.path.split('?')[1] if '?' in self.path else ""
            params = parse_qs(query)
            
            if 'code' in params:
                auth_code = params['code'][0]
                self.send_response(200)
                self.send_header('Content-type', 'text/html; charset=utf-8')
                self.end_headers()
                html = """
                <html>
                <head><title>Spotify Authorization Success</title></head>
                <body style="font-family: Arial; text-align: center; padding: 50px;">
                    <h2>✅ Cấp quyền thành công!</h2>
                    <p>Bạn có thể đóng cửa sổ này và quay lại Jupyter.</p>
                </body>
                </html>
                """
                self.wfile.write(html.encode('utf-8'))
                print("\n✅ Nhận được authorization code từ Spotify!")
            else:
                self.send_response(400)
                self.end_headers()
                error = params.get('error', ['unknown'])[0]
                print(f"\n❌ Lỗi: {error}")
    
    def log_message(self, format, *args):
        pass  # Tắt log mặc định

# ===== Bước 3: Chạy server callback trong background =====
print("\n🔄 Chuẩn bị server callback...")

try:
    server = HTTPServer(("127.0.0.1", 8888), CallbackHandler)
    server_thread = threading.Thread(target=server.serve_forever, daemon=True)
    server_thread.start()
    server_ready = True
    print("✅ Server callback đã sẵn sàng trên http://127.0.0.1:8888")
except Exception as e:
    print(f"❌ Lỗi khởi động server: {e}")
    print("   Có thể port 8888 đã được sử dụng. Đóng các ứng dụng khác hoặc đổi port.")

# ===== Bước 4: Mở URL authorization =====
if server_ready and client_id and client_secret:
    auth_url = get_authorization_url(client_id, REDIRECT_URI, SCOPES)
    
    print("\n🌐 Mở trình duyệt để cấp quyền...")
    print(f"📌 Hoặc sao chép URL này: {auth_url}")
    
    # Cố gắng mở trình duyệt
    try:
        webbrowser.open(auth_url)
        print("\n⏳ Đợi người dùng cấp quyền... (tối đa 60 giây)")
        
        # Đợi tối đa 60 giây để nhận auth code
        for i in range(60):
            if auth_code:
                break
            time.sleep(1)
        
        if auth_code:
            print(f"\n✅ Nhận được authorization code: {auth_code[:20]}...{auth_code[-10:]}")
        else:
            print("\n⏱️ Hết giờ chờ. Vui lòng chạy cell tiếp theo để hoàn thành flow.")
    except Exception as e:
        print(f"⚠️ Không thể mở trình duyệt: {e}")
        print(f"📌 Vui lòng sao chép URL này vào trình duyệt: {auth_url}")
else:
    print("⚠️ Chưa sẵn sàng. Kiểm tra credentials hoặc port 8888.")


In [ ]:

# ===== Bước 5: Đổi Authorization Code thành Access Token =====

def get_user_access_token(auth_code, client_id, client_secret, redirect_uri):
    """Đổi authorization code thành user access token"""
    token_url = 'https://accounts.spotify.com/api/token'
    
    # Tạo Basic Auth header
    auth_str = f"{client_id}:{client_secret}"
    auth_bytes = auth_str.encode('utf-8')
    auth_b64 = base64.b64encode(auth_bytes).decode('utf-8')
    
    headers = {
        'Authorization': f'Basic {auth_b64}',
        'Content-Type': 'application/x-www-form-urlencoded'
    }
    
    data = {
        'grant_type': 'authorization_code',
        'code': auth_code,
        'redirect_uri': redirect_uri
    }
    
    try:
        response = requests.post(token_url, headers=headers, data=data, timeout=10)
        response.raise_for_status()
        json_result = response.json()
        
        return {
            'access_token': json_result.get('access_token'),
            'refresh_token': json_result.get('refresh_token'),
            'expires_in': json_result.get('expires_in'),
            'token_type': json_result.get('token_type')
        }
    except requests.exceptions.RequestException as e:
        print(f"❌ Lỗi đổi token: {e}")
        return None

# ===== Xử lý Authorization Code Flow =====
if auth_code:
    print("\n🔐 Đang đổi authorization code thành access token...")
    
    token_response = get_user_access_token(auth_code, client_id, client_secret, REDIRECT_URI)
    
    if token_response:
        user_access_token = token_response['access_token']
        refresh_token = token_response['refresh_token']
        expires_in = token_response['expires_in']
        
        print(f"✅ Lấy user access token thành công!")
        print(f"   Access Token: {user_access_token[:30]}...{user_access_token[-10:]}")
        print(f"   Refresh Token: {refresh_token[:20] if refresh_token else 'N/A'}...{refresh_token[-10:] if refresh_token else ''}")
        print(f"   Hạn sử dụng: {expires_in} giây (~{expires_in//3600} giờ)")
        
        # Tạo Spotify client với user access token
        print("\n📚 Khởi tạo Spotipy client với user token...")
        sp_user = spotipy.Spotify(auth=user_access_token)
        
        # Test: Lấy thông tin người dùng
        try:
            current_user = sp_user.current_user()
            print(f"✅ Kết nối thành công!")
            print(f"   Người dùng: {current_user.get('display_name', 'Anonymous')}")
            print(f"   Email: {current_user.get('email', 'N/A')}")
            print(f"   ID: {current_user.get('id', 'N/A')}")
        except Exception as e:
            print(f"⚠️ Lỗi lấy thông tin người dùng: {e}")
    else:
        print("❌ Không thể lấy access token.")
else:
    print("⚠️ Chưa có authorization code. Vui lòng chạy cell trước để hoàn thành authorization flow.")

# Đóng server callback
if 'server' in locals():
    server.shutdown()
    print("\n🛑 Server callback đã đóng.")


In [ ]:

# ===== PHƯƠNG ÁN KHÁC: Sử dụng User Access Token (Authorization Code Flow) =====
# Nếu muốn dùng user token từ cell 11, chạy cell này thay vì cell trên

import time

def fetch_audio_features_with_user_token(track_ids, user_token, batch_size=100):
    """
    Lấy audio features sử dụng user access token từ Authorization Code Flow
    """
    all_features = []
    print(f"\n📊 Bắt đầu lấy audio features (USER TOKEN) cho {len(track_ids)} tracks...")
    print(f"   Batch size: {batch_size} tracks/request")
    
    # Tạo Spotify client với user token
    sp_user = spotipy.Spotify(auth=user_token)
    
    # Chia batch 100
    for i in range(0, len(track_ids), batch_size):
        batch = track_ids[i:i+batch_size]
        batch_num = i // batch_size + 1
        total_batches = (len(track_ids) + batch_size - 1) // batch_size
        
        try:
            results = sp_user.audio_features(batch)
            
            # Lọc bỏ giá trị None
            valid_features = [f for f in results if f is not None]
            all_features.extend(valid_features)
            
            print(f"✅ Batch {batch_num}/{total_batches} - Lấy được {len(valid_features)} tracks")
            
            # Nghỉ để tránh rate limit
            time.sleep(0.1)
            
        except spotipy.exceptions.SpotifyException as e:
            if e.http_status == 403:
                print(f"\n❌ LỖI 403 vẫn xảy ra (User Management)")
                print(">>> Vui lòng vào Spotify Developer Dashboard")
                print(">>> Mục 'Settings' > 'User Management' > Thêm email")
                return all_features
            elif e.http_status == 429:
                wait_time = int(e.headers.get('Retry-After', 5))
                print(f"⏸️ Rate Limit! Đang chờ {wait_time}s...")
                time.sleep(wait_time)
                continue
            else:
                print(f"⚠️ Lỗi API tại batch {batch_num}: {e}")
    
    return all_features

# ===== THỰC THI =====
if 'user_access_token' in locals() and user_access_token:
    print("🔐 Đang sử dụng USER ACCESS TOKEN từ Authorization Code Flow...")
    
    track_ids_list = df_merged['track_id'].unique().tolist()
    
    # Gọi hàm với user token
    audio_features_list = fetch_audio_features_with_user_token(track_ids_list, user_access_token)
    
    if audio_features_list:
        df_features = pd.DataFrame(audio_features_list)
        print(f"\n✅ Tổng kết: Lấy thành công {len(df_features)}/{len(track_ids_list)} tracks.")
        print(f"   Cột dữ liệu: {list(df_features.columns)}")
    else:
        print("\n⚠️ Không thu thập được dữ liệu.")
else:
    print("⚠️ Chưa có user_access_token. Vui lòng chạy Authorization Code Flow (Cell 10-11) trước.")


In [ ]:

# ===== PHƯƠNG ÁN KHẨN CẤP: Tạo Mock Audio Features (Nếu API bị chặn) =====
# Nếu vẫn gặp lỗi 403, bạn có thể dùng dữ liệu giả này để tiếp tục

import numpy as np

print("\n" + "="*70)
print("PHƯƠNG ÁN KHẨN CẤP: TẠO DỮ LIỆU AUDIO FEATURES GIẢ")
print("="*70)

def create_mock_audio_features(track_ids):
    """
    Tạo dữ liệu audio features giả để phát triển/test
    Bạn có thể xóa hàm này khi API Spotify hoạt động bình thường
    """
    print(f"\n⚠️ Tạo {len(track_ids)} bản ghi audio features giả...")
    
    mock_data = []
    audio_feature_names = [
        'danceability', 'energy', 'key', 'loudness', 'mode',
        'speechiness', 'acousticness', 'instrumentalness', 'liveness',
        'valence', 'tempo', 'time_signature'
    ]
    
    for track_id in track_ids:
        # Tạo giá trị ngẫu nhiên hợp lý
        feature_dict = {
            'id': track_id,
            'uri': f'spotify:track:{track_id}',
            'track_href': f'https://api.spotify.com/v1/tracks/{track_id}',
            'analysis_url': f'https://api.spotify.com/v1/audio-analysis/{track_id}',
            'type': 'audio_features',
            'danceability': np.random.uniform(0, 1),
            'energy': np.random.uniform(0, 1),
            'key': np.random.randint(0, 12),
            'loudness': np.random.uniform(-60, 0),
            'mode': np.random.randint(0, 2),
            'speechiness': np.random.uniform(0, 1),
            'acousticness': np.random.uniform(0, 1),
            'instrumentalness': np.random.uniform(0, 1),
            'liveness': np.random.uniform(0, 1),
            'valence': np.random.uniform(0, 1),
            'tempo': np.random.uniform(60, 200),
            'time_signature': np.random.choice([3, 4, 5])
        }
        mock_data.append(feature_dict)
    
    print(f"✅ Tạo xong {len(mock_data)} bản ghi dữ liệu giả")
    return mock_data

# ===== CHỌN PHƯƠNG ÁN: API thực hay dữ liệu giả =====
if 'user_access_token' in locals() and user_access_token:
    print("\n🔐 Dùng USER ACCESS TOKEN từ Authorization Code Flow...")
    
    track_ids_list = df_merged['track_id'].unique().tolist()
    
    try:
        # Cố gắng lấy từ API
        audio_features_list = fetch_audio_features_with_user_token(track_ids_list, user_access_token)
        
        if not audio_features_list or len(audio_features_list) == 0:
            print("\n❌ API không trả về dữ liệu, sử dụng mock data...")
            audio_features_list = create_mock_audio_features(track_ids_list)
    except Exception as e:
        print(f"\n❌ Lỗi gọi API: {e}")
        print("💾 Sử dụng mock data thay thế...")
        audio_features_list = create_mock_audio_features(track_ids_list)
else:
    print("\n⚠️ Không có user access token, sử dụng mock data...")
    track_ids_list = df_merged['track_id'].unique().tolist()
    audio_features_list = create_mock_audio_features(track_ids_list)

# ===== Tạo DataFrame từ dữ liệu =====
if audio_features_list and len(audio_features_list) > 0:
    df_features = pd.DataFrame(audio_features_list)
    print(f"\n✅ Tổng kết: Có {len(df_features)}/{len(track_ids_list)} tracks")
    print(f"   Cột dữ liệu: {list(df_features.columns)}")
    print(f"\n📊 Mẫu dữ liệu:")
    print(df_features.head())
else:
    print("\n❌ Không thể tạo dữ liệu features")


In [ ]:
# Bước 4: Gộp dữ liệu
if not df_features.empty:
    print("\n🔗 Gộp dữ liệu...")
    
    # Đổi tên cột 'id' thành 'track_id' để match với df_merged
    df_features_renamed = df_features.rename(columns={'id': 'track_id'})
    
    # Merge
    df_final = df_merged.merge(df_features_renamed, on='track_id', how='left', suffixes=('', '_audio'))
    
    print(f"✅ Gộp thành công!")
    print(f"   Shape gốc: {df_merged.shape}")
    print(f"   Shape features: {df_features_renamed.shape}")
    print(f"   Shape final: {df_final.shape}")
    print(f"   Cột mới thêm: {df_features_renamed.shape[1] - 1}")
    
    # Lưu file
    output_path = '../data/processed/spotify_merged_with_features.csv'
    os.makedirs('../data/processed', exist_ok=True)
    df_final.to_csv(output_path, index=False)
    print(f"\n💾 Đã lưu tại: {os.path.abspath(output_path)}")
    
    # Hiển thị mẫu
    print(f"\n📊 Mẫu dữ liệu:")
    print(df_final.head(2))
else:
    print("\n⚠️  Không có dữ liệu features để gộp")